# Lab Task B1: Custom Max Pooling Aggregation

### Task Description
Implement a Transformer classifier that uses **Max Pooling** across the sequence dimension instead of Mean Pooling to aggregate token representations.

### Instructions
1. Implement `MaxPoolingTransformer.__init__()` and `MaxPoolingTransformer.forward()`.
2. Apply `torch.max()` along the sequence length dimension (`dim=1`) to extract maximum activations across tokens for each feature dimension.

In [5]:
import torch
import torch.nn as nn
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# =============================================================================
# Positional Encoding
# =============================================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)

        # Position: 0, 1, 2, ..., max_len-1
        position = torch.arange(
            0, max_len, dtype=torch.float
        ).unsqueeze(1)

        # Frequency values
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )

        # Even positions use sine
        pe[:, 0::2] = torch.sin(position * div_term)

        # Odd positions use cosine
        pe[:, 1::2] = torch.cos(position * div_term)

        # Add batch dimension
        pe = pe.unsqueeze(0)

        # Store as buffer, not as trainable parameter
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]
    
class MaxPoolingTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=64, nhead=2, num_layers=2, max_len=32):
        super().__init__()
        self.d_model = d_model
        
        # =========================================================================
        # TODO: Implement model layers for Max Pooling Classifier
        # 1. Embedding layer (padding_idx=0)
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        # 2. PositionalEncoding instance
        self.pos_encoder = PositionalEncoding(d_model=d_model, max_len=max_len)
        # 3. TransformerEncoderLayer and TransformerEncoder (batch_first=True)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=256, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        # 4. Linear classifier head (d_model -> 1)
        self.classifier = nn.Linear(d_model, 1)
        # =========================================================================
        
        # YOUR CODE HERE

    def forward(self, x, padding_mask=None):
        # =========================================================================
        # TODO: Implement forward pass with Max Pooling
        # 1. Embed x and scale by sqrt(d_model)
        embedded = self.embedding(x)
        embedded = embedded * math.sqrt(self.d_model)
        # 2. Add positional encoding
        embedded = self.pos_encoder(embedded)
        # 3. Pass through transformer encoder with padding_mask
        encoded = self.transformer_encoder(embedded, src_key_padding_mask = padding_mask)
        # 4. Perform Max Pooling across sequence length (dim=1) using torch.max().values
        pooled = torch.max(encoded, dim = 1).values
        # 5. Compute logits through linear classifier head and squeeze output
        # =========================================================================
        logits = self.classifier(pooled)
        logits = logits.squeeze(-1)
        return logits
        # YOUR CODE HERE

# =============================================================================
# Automated Verification Suite (Do not modify)
# =============================================================================
def test_task_B1():
    torch.manual_seed(42)
    dummy_vocab_size = 60
    batch_size, seq_len = 8, 10
    
    dummy_input = torch.randint(1, dummy_vocab_size, (batch_size, seq_len)).to(device)
    dummy_mask = (dummy_input == 0).to(device)
    
    model = MaxPoolingTransformer(vocab_size=dummy_vocab_size, d_model=64).to(device)
    
    try:
        logits = model(dummy_input, padding_mask=dummy_mask)
        assert logits is not None, "Forward pass returned None."
        assert logits.shape == torch.Size([batch_size]), f"Expected shape torch.Size([{batch_size}]), got {logits.shape}"
        
        print(f"Test 1 Passed: Max pooling output matches shape ({batch_size},).")
        print("Lab task 4 passed!")
    except Exception as e:
        print(f"Test Failed: {e}")

test_task_B1()

Test 1 Passed: Max pooling output matches shape (8,).
Lab task 4 passed!
